# Short-Term LSTM Attention Experiment

This Colab notebook compares the current short-term LSTM design against an attention variant.

Both models use the same clean gold dataset, temporal split, sequence construction, scaling, loss, optimizer, and restored target metrics. The only intended architectural difference is how the sequence branch summarizes hidden states before the dense output head.

- Baseline model: use the final LSTM hidden state.
- Attention model: compute learned attention weights over all LSTM hidden states, then use the weighted context vector.

The decision rule is validation-first: use validation MAE to decide whether the attention variant is worth promoting to local source, then inspect test metrics as holdout evidence.


In [ ]:
# Colab setup: mount Drive, install lightweight dependencies, and make project source importable.
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = Path("/content").exists()
DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_PROJECT_DIR = DRIVE_ROOT / "nba-scout-assistant"
COLAB_REPO_DIR = Path("/content/nba-scout-assistant")
REPO_URL = "https://github.com/kdnehihi/nba-scout-assistant.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mlflow", "scikit-learn", "pandas", "pyarrow", "torch"], check=True)

PROJECT_ROOT_CANDIDATES = [
    DRIVE_PROJECT_DIR,
    COLAB_REPO_DIR,
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve(),
]

PROJECT_ROOT = next((path for path in PROJECT_ROOT_CANDIDATES if (path / "src").exists()), None)

if PROJECT_ROOT is None and IN_COLAB:
    subprocess.run(["git", "clone", REPO_URL, str(COLAB_REPO_DIR)], check=True)
    PROJECT_ROOT = COLAB_REPO_DIR

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find project source directory containing src/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = DRIVE_PROJECT_DIR / "data" if IN_COLAB else PROJECT_ROOT / "data"
MLFLOW_DB_PATH = DRIVE_PROJECT_DIR / "mlflow.db" if IN_COLAB else PROJECT_ROOT / "mlflow.db"
MLFLOW_ARTIFACT_DIR = DRIVE_PROJECT_DIR / "mlartifacts" if IN_COLAB else PROJECT_ROOT / "mlartifacts"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("MLFLOW_DB_PATH:", MLFLOW_DB_PATH)
print("MLFLOW_ARTIFACT_DIR:", MLFLOW_ARTIFACT_DIR)


In [ ]:
from __future__ import annotations

from copy import deepcopy
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader

from src.config.lstm_config import resolve_lstm_task_config
from src.dataset.loaders import load_performance_training_clean, resolve_data_paths
from src.dataset.lstm_dataset import ShortTermLSTMDataset
from src.dataset.scaling import scale_lstm_inputs
from src.dataset.sequence import make_lstm_delta_sequences, prepare_sequence_training
from src.evaluation.evaluate_short_term import evaluate_lstm_predictions_by_split, predict_lstm_actuals
from src.models.lstm import ShortTermLSTM
from src.training.mlflow_utils import configure_mlflow

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

def set_seed(seed: int = SEED) -> None:
    """Input: random seed. Output: deterministic-ish torch/numpy/random state."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print("DEVICE:", DEVICE)


## Data Loading

The experiment reads `performance_training_clean.parquet` from the Drive data folder:

```text
/content/drive/MyDrive/nba-scout-assistant/data/gold/performance_training_clean.parquet
```


In [ ]:
paths = resolve_data_paths(DATA_DIR)
performance_path = paths.gold_dir / "performance_training_clean.parquet"
print("performance_path:", performance_path)
print("exists:", performance_path.exists())

performance = load_performance_training_clean(paths)
print("performance", performance.shape)
print(performance["split"].value_counts().sort_index())


## Model Definitions

The baseline model is imported from local source. The attention variant keeps the same static branch and dense head idea, but replaces `hidden[-1]` with an attention-weighted context across all LSTM output states.

In [ ]:
class ShortTermAttentionLSTM(nn.Module):
    """LSTM with additive attention over hidden states plus static context fusion."""

    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        static_size: int,
        static_hidden_size: int = 16,
        fusion_hidden_size: int = 32,
        attention_hidden_size: int = 32,
        dropout: float = 0.15,
    ):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
        )
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, attention_hidden_size),
            nn.Tanh(),
            nn.Linear(attention_hidden_size, 1),
        )
        self.sequence_dropout = nn.Dropout(dropout)
        self.static_encoder = nn.Sequential(
            nn.Linear(static_size, static_hidden_size),
            nn.ReLU(),
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size + static_hidden_size, fusion_hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden_size, 1),
        )

    def forward(self, x_seq: torch.Tensor, x_static: torch.Tensor) -> torch.Tensor:
        """Return predicted delta from attention-weighted sequence context and static inputs."""
        sequence_output, _ = self.lstm(x_seq)
        attention_logits = self.attention(sequence_output).squeeze(-1)
        attention_weights = torch.softmax(attention_logits, dim=1)
        context = torch.sum(sequence_output * attention_weights.unsqueeze(-1), dim=1)
        context = self.sequence_dropout(context)
        static_context = self.static_encoder(x_static)
        combined = torch.cat([context, static_context], dim=1)
        prediction = self.fc(combined)
        return prediction.squeeze(1)

## Training Helpers

Metrics are computed on restored next-five-game averages, not on scaled or delta-space loss.

In [ ]:
def train_one_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module, optimizer: Adam) -> float:
    """Input: model and train loader. Output: sample-weighted training loss."""
    model.train()
    total_loss = 0.0
    total_samples = 0
    for x_seq, x_static, y in loader:
        x_seq = x_seq.to(DEVICE)
        x_static = x_static.to(DEVICE)
        y = y.to(DEVICE)
        optimizer.zero_grad()
        prediction = model(x_seq, x_static)
        loss = criterion(prediction, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        batch_size = y.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size
    return total_loss / total_samples


def evaluate_loss(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> float:
    """Input: model and eval loader. Output: sample-weighted eval loss."""
    model.eval()
    total_loss = 0.0
    total_samples = 0
    with torch.no_grad():
        for x_seq, x_static, y in loader:
            x_seq = x_seq.to(DEVICE)
            x_static = x_static.to(DEVICE)
            y = y.to(DEVICE)
            prediction = model(x_seq, x_static)
            loss = criterion(prediction, y)
            batch_size = y.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size
    return total_loss / total_samples


def build_task_arrays(df: pd.DataFrame, task_config):
    """Input: performance table and task config. Output: scaled arrays plus metadata for one task."""
    sequence_df = prepare_sequence_training(df, task_config)
    X_seq, X_static, y_delta, y_actual, baseline, split = make_lstm_delta_sequences(sequence_df, task_config)
    train_mask = split == "train"
    X_seq_scaled, X_static_scaled, y_delta_model, seq_scaler, static_scaler, y_scaler = scale_lstm_inputs(
        X_seq,
        X_static,
        y_delta,
        train_mask,
        scale_target_delta=task_config.scale_target_delta,
    )
    return {
        "X_seq": X_seq,
        "X_static": X_static,
        "X_seq_scaled": X_seq_scaled,
        "X_static_scaled": X_static_scaled,
        "y_delta_model": y_delta_model,
        "y_actual": y_actual,
        "baseline": baseline,
        "split": split,
        "y_scaler": y_scaler,
    }


def make_loaders(arrays: dict[str, np.ndarray], batch_size: int):
    """Input: task arrays. Output: train, validation, and test dataloaders."""
    split = arrays["split"]
    loaders = {}
    for split_name, shuffle in [("train", True), ("validation", False), ("test", False)]:
        mask = split == split_name
        dataset = ShortTermLSTMDataset(
            arrays["X_seq_scaled"][mask],
            arrays["X_static_scaled"][mask],
            arrays["y_delta_model"][mask],
        )
        loaders[split_name] = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)
    return loaders

In [ ]:
def train_model_variant(
    task_name: str,
    model_name: str,
    model: nn.Module,
    arrays: dict[str, np.ndarray],
    task_config,
    max_epochs: int = 100,
    patience: int = 8,
    min_delta: float = 1e-4,
) -> tuple[nn.Module, pd.DataFrame, dict[str, float]]:
    """Input: model variant and task arrays. Output: best model, restored metrics, and training diagnostics."""
    loaders = make_loaders(arrays, batch_size=task_config.batch_size)
    model = model.to(DEVICE)
    criterion = nn.HuberLoss(delta=1.0)
    optimizer = Adam(model.parameters(), lr=task_config.learning_rate)

    best_val_loss = float("inf")
    best_state = None
    stale_epochs = 0
    history = []

    for epoch in range(max_epochs):
        train_loss = train_one_epoch(model, loaders["train"], criterion, optimizer)
        val_loss = evaluate_loss(model, loaders["validation"], criterion)
        history.append({"epoch": epoch + 1, "train_loss": train_loss, "validation_loss": val_loss})

        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            best_state = deepcopy(model.state_dict())
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    y_pred = predict_lstm_actuals(
        model=model,
        X_seq_scaled=arrays["X_seq_scaled"],
        X_static_scaled=arrays["X_static_scaled"],
        baseline=arrays["baseline"],
        y_scaler=arrays["y_scaler"],
        batch_size=task_config.batch_size,
        device=DEVICE,
    )
    metrics = evaluate_lstm_predictions_by_split(
        y_actual=arrays["y_actual"],
        y_pred=y_pred,
        split=arrays["split"],
        task=task_name,
        splits=("validation", "test"),
    )
    metrics.insert(0, "model", model_name)

    diagnostics = {
        "best_validation_loss": best_val_loss,
        "epochs_ran": len(history),
    }
    return model, metrics, diagnostics

## Run Experiment

Default runs all three tasks. For quick smoke tests, set `TASKS = ["points"]` and lower `MAX_EPOCHS`.

In [ ]:
TASKS = ["points", "assists", "rebounds"]
MAX_EPOCHS = 100
PATIENCE = 8
MIN_DELTA = 1e-4

mlflow_client = configure_mlflow(
    tracking_db_path=MLFLOW_DB_PATH,
    artifact_dir=MLFLOW_ARTIFACT_DIR,
    experiment_name="nba_scout_lstm_attention_experiment",
)

all_metrics = []
all_diagnostics = []

for task in TASKS:
    set_seed(SEED)
    task_name, task_config = resolve_lstm_task_config(task)
    arrays = build_task_arrays(performance, task_config)
    input_size = arrays["X_seq"].shape[-1]
    static_size = arrays["X_static"].shape[-1]

    variants = {
        "current_lstm": ShortTermLSTM(
            input_size=input_size,
            hidden_size=task_config.hidden_size,
            static_size=static_size,
            dropout=task_config.dropout,
        ),
        "attention_lstm": ShortTermAttentionLSTM(
            input_size=input_size,
            hidden_size=task_config.hidden_size,
            static_size=static_size,
            dropout=task_config.dropout,
        ),
    }

    for model_name, model in variants.items():
        set_seed(SEED)
        with mlflow_client.start_run(run_name=f"{model_name}_{task_name}"):
            mlflow_client.log_params({
                "task": task_name,
                "model": model_name,
                "sequence_length": task_config.sequence_length,
                "hidden_size": task_config.hidden_size,
                "batch_size": task_config.batch_size,
                "learning_rate": task_config.learning_rate,
                "dropout": task_config.dropout,
                "loss": "HuberLoss_delta_1.0",
                "scale_target_delta": task_config.scale_target_delta,
                "max_epochs": MAX_EPOCHS,
                "patience": PATIENCE,
                "min_delta": MIN_DELTA,
                "seed": SEED,
            })
            _, metrics, diagnostics = train_model_variant(
                task_name=task_name,
                model_name=model_name,
                model=model,
                arrays=arrays,
                task_config=task_config,
                max_epochs=MAX_EPOCHS,
                patience=PATIENCE,
                min_delta=MIN_DELTA,
            )
            for _, row in metrics.iterrows():
                prefix = row["split"]
                mlflow_client.log_metric(f"{prefix}_rows", int(row["rows"]))
                mlflow_client.log_metric(f"{prefix}_mae", float(row["mae"]))
                mlflow_client.log_metric(f"{prefix}_rmse", float(row["rmse"]))
                mlflow_client.log_metric(f"{prefix}_r2", float(row["r2"]))
            mlflow_client.log_metric("best_validation_loss", float(diagnostics["best_validation_loss"]))
            mlflow_client.log_metric("epochs_ran", int(diagnostics["epochs_ran"]))

        metrics["epochs_ran"] = diagnostics["epochs_ran"]
        metrics["best_validation_loss"] = diagnostics["best_validation_loss"]
        all_metrics.append(metrics)
        all_diagnostics.append({"task": task_name, "model": model_name, **diagnostics})
        print(task_name, model_name, diagnostics)

attention_evaluation = pd.concat(all_metrics, ignore_index=True)
attention_evaluation.sort_values(["task", "split", "mae"])

## Selection View

The preferred model per task is selected by validation MAE. Test metrics are shown for audit after validation selection.

In [ ]:
validation_rank = (
    attention_evaluation[attention_evaluation["split"].eq("validation")]
    .sort_values(["task", "mae"])
    .groupby("task", as_index=False)
    .first()[["task", "model", "mae", "rmse", "r2", "epochs_ran"]]
    .rename(columns={"model": "selected_by_validation", "mae": "validation_mae", "rmse": "validation_rmse", "r2": "validation_r2"})
)

test_audit = attention_evaluation[attention_evaluation["split"].eq("test")][["task", "model", "mae", "rmse", "r2"]]
selection_summary = validation_rank.merge(
    test_audit,
    left_on=["task", "selected_by_validation"],
    right_on=["task", "model"],
    how="left",
).drop(columns=["model"]).rename(columns={"mae": "selected_test_mae", "rmse": "selected_test_rmse", "r2": "selected_test_r2"})

selection_summary

In [ ]:
comparison = attention_evaluation.pivot_table(
    index=["task", "split"],
    columns="model",
    values="mae",
).reset_index()
if {"attention_lstm", "current_lstm"}.issubset(comparison.columns):
    comparison["attention_minus_current_mae"] = comparison["attention_lstm"] - comparison["current_lstm"]
comparison

## Promotion Rule

Promote attention to local source only if it improves validation MAE for the target task without a clear test degradation. If gains are small or inconsistent, keep the current LSTM and treat attention as an explored but unselected architecture.